In [2]:
import os
import pandas as pd

In [3]:
# Define input filenames
GENE_POSITIONS_FILE = "Rest_genes.csv"
MASTER_DATA_FILE = "IIT jammu/DataSet_variants.tsv"
CHUNK_SIZE = 100000  # Process 100k rows at a time to keep RAM footprint low

In [4]:
print("Step 1: Loading strict gene positions from CSV...")

# 1. Read the positions file strictly as provided
gene_df = pd.read_csv(GENE_POSITIONS_FILE)

# 2. Parse the positions array dynamically out of your specific layout
# Structure of "Genomic Position (Start to End)" column is: "START to END"
gene_intervals = []
for _, row in gene_df.iterrows():
    gene_name = str(row["Gene Name"]).strip()
    chrom = str(row["Chromosome No."]).strip()
    
    # Split the position string by literal " to "
    pos_string = str(row["Genomic Position (Start to End)"])
    start_bp, end_bp = map(int, pos_string.lower().split(" to "))
    
    gene_intervals.append({
        "name": gene_name,
        "chrom": chrom,
        "start": start_bp,
        "end": end_bp,
        "out_file": f"{gene_name}_variants.csv",
        "initialized": False
    })

print(f"Loaded {len(gene_intervals)} target genes successfully.")

Step 1: Loading strict gene positions from CSV...
Loaded 4 target genes successfully.


In [5]:
print("Step 2: Commencing streaming extraction pass through the 9 GB master file...")

# 3. Stream the master file in chunks using an iterator
# Assuming Column 1 = Chromosome, Column 2 = Base Pair Position
chunk_count = 0
for chunk in pd.read_csv(MASTER_DATA_FILE, sep="\t", chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    
    # Standardize data types of indexing columns to avoid text matching errors
    chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
    chunk.iloc[:, 1] = pd.to_numeric(chunk.iloc[:, 1], errors="coerce")
    
    # Evaluate each gene interval against the current chunk matrix
    for gene in gene_intervals:
        # Create a boolean mask matching your strict chromosome and boundary window
        mask = (chunk.iloc[:, 0] == gene["chrom"]) & \
               (chunk.iloc[:, 1] >= gene["start"]) & \
               (chunk.iloc[:, 1] <= gene["end"])
        
        matched_rows = chunk[mask]
        
        # If any variants fall within the window, save them out
        if not matched_rows.empty:
            # If this is the very first match for this gene, write out headers ('w' mode)
            if not gene["initialized"]:
                matched_rows.to_csv(gene["out_file"], index=False, mode="w")
                gene["initialized"] = True
            else:
                # Otherwise, cleanly append to the existing file without printing headers again
                matched_rows.to_csv(gene["out_file"], index=False, mode="a", header=False)
                
    if chunk_count % 10 == 0:
        print(f"Processed {chunk_count * CHUNK_SIZE / 1000000:.1f} Million rows...")

print("\nSuccess! All target gene variants sliced out safely with zero RAM overflow.")

Step 2: Commencing streaming extraction pass through the 9 GB master file...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0        1
1        1
2        1
3        1
4        1
        ..
99995    1
99996    1
99997    1
99998    1
99999    1
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '100000    1
100001    1
100002    1
100003    1
100004    1
         ..
199995    1
199996    1
199997    1
199998    1
199999    1
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].ast

Processed 1.0 Million rows...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1000000    1
1000001    1
1000002    1
1000003    1
1000004    1
          ..
1099995    1
1099996    1
1099997    1
1099998    1
1099999    1
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1100000    1
1100001    1
1100002    1
1100003    1
1100004    1
          ..
1199995    1
1199996    1
1199997    1
1199998    1
1199999    1
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.

Processed 2.0 Million rows...


...

C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '144100000    23
144100001    23
144100002    23
144100003    23
144100004    23
             ..
144199995    23
144199996    23
144199997    23
144199998    23
144199999    23
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '144200000    23
144200001    23
144200002    23
144200003    23
144200004    23
             ..
144299995    23
144299996    23
144299997    23
144299998    23
144299999    23
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with i

Processed 145.0 Million rows...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '145100000    23
145100001    23
145100002    23
145100003    23
145100004    23
             ..
145199995    23
145199996    23
145199997    23
145199998    23
145199999    23
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '145200000    23
145200001    23
145200002    23
145200003    23
145200004    23
             ..
145299995    23
145299996    23
145299997    23
145299998    23
145299999    23
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with i

Processed 146.0 Million rows...

Success! All target gene variants sliced out safely with zero RAM overflow.


C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '146000000    23
146000001    23
146000002    23
146000003    23
146000004    23
             ..
146099995    23
146099996    23
146099997    23
146099998    23
146099999    23
Name: chromosome, Length: 100000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2704\3559725011.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '146100000    23
146100001    23
146100002    23
146100003    23
146100004    23
             ..
146168253    23
146168254    23
146168255    23
146168256    23
146168257    23
Name: chromosome, Length: 68258, dtype: object' has dtype incompatible with in